# Maia Lite 0.3.0 — preparação segura do pré-treinamento 0,5B

Este notebook **não** inicia treinamento real nem altera flags de direitos. Verifica estado, GPU e prepara evidências. O treino com corpus integral requer aprovação e shards finalizados.

In [ ]:
from pathlib import Path
import json, hashlib
import torch
ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full/recovery_audit_v4')
EXPECTED_INVENTORY_SHA = 'ad19d1d98e2cf5b91e650f343e6430bbde34b549b5fbc352b47682a91d7ab562'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
 print('GPU:', torch.cuda.get_device_name(0))
 print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print('Root exists:', ROOT.exists())


In [ ]:
PROVENANCE = ROOT/'production_resume_v2/wikimedia_rights_review/provenance_audit/provenance_summary.json'
assert PROVENANCE.is_file(), 'Monte o Google Drive e confirme o caminho do relatório de proveniência.'
s = json.loads(PROVENANCE.read_text())
print('Provenance status:', s.get('status'))
print('Chunks:',s.get('scanned_chunks'),'/',s.get('total_remaining_chunks'))
print('Records:',s.get('scanned_records'))
assert s.get('inventory_sha256') == EXPECTED_INVENTORY_SHA, 'Inventário divergente: não prosseguir.'
assert s.get('scanned_chunks') == s.get('total_remaining_chunks') == 650, 'Auditoria incompleta.'


In [ ]:
# Diagnóstico: NÃO altera aprovações e NÃO permite bypass.
STATUS = ROOT/'production_resume_v2/wikimedia_rights_review/provenance_audit/provenance_summary.json'
status = json.loads(STATUS.read_text())
ready = (status.get('human_rights_approval') is True and
         status.get('snapshot_verified') is True and
         status.get('third_party_exceptions_cleared') is True)
print('Rights gate ready:',ready)
if not ready:
 print('BLOCKED: revisão de direitos pendente. Nenhum treinamento com estes chunks será iniciado.')
print('Próxima etapa permitida: dimensionamento do modelo e smoke test sintético.')
